# Practica 2 - Estadistica descriptiva

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

path = "../data/procesed/metro.csv"
metro = pd.read_csv(path)
metro.head()

In [ ]:
metro.info()
metro["date_time"] = pd.to_datetime(metro["date_time"])


## Resumen general

In [ ]:
metro.describe()

## Tendencia central

In [ ]:
columnas_numericas = metro.select_dtypes(include=[np.number]).columns

media = metro[columnas_numericas].mean()
mediana = metro[columnas_numericas].median()
moda_num = metro[columnas_numericas].mode().iloc[0]

resumen_tendencia_central = pd.DataFrame({
    "Media": media,
    "mediana": mediana,
    "moda": moda_num
})

print("medidas tendencia central:")
print(resumen_tendencia_central)

print("")
print("moda variables categoricas:")
print("moda de weather_main:", metro["weather_main"].mode()[0])
print("moda de weather_description:", metro["weather_description"].mode()[0])

## Dispersion

In [ ]:
varianza = metro[columnas_numericas].var()
desviacion_estandar = metro[columnas_numericas].std()
minimo = metro[columnas_numericas].min()
maximo = metro[columnas_numericas].max()
rango = maximo - minimo

q1 = metro[columnas_numericas].quantile(0.25)
q3 = metro[columnas_numericas].quantile(0.75)
iqr = q3 - q1

coeficiente_variacion = (desviacion_estandar / media.abs()) * 100

resumen_dispersion = pd.DataFrame({
    "varianza": varianza,
    "desviacion Estandar": desviacion_estandar,
    "Minimo": minimo,
    "maximo": maximo,
    "Rango": rango,
    "IQR": iqr,
    "cV (%)": coeficiente_variacion
})

resumen_dispersion

## Forma y posicion

In [ ]:
asimetria = metro[columnas_numericas].skew().sort_values()

curtosis = metro[columnas_numericas].kurt().sort_values()

print("skewness:")
print(asimetria)

print("")
print("kurtosis:")
print(curtosis)

cuartiles_df = metro[columnas_numericas].quantile([0.10, 0.25, 0.50, 0.75, 0.90]).T
cuartiles_df.columns = ["p10", "Q1 (P25)", "Q2 (Mediana)", "Q3 (P75)", "P90"]
print("")
print("percentiles y cuartiles")
print(cuartiles_df)

## Variables categoricas

In [ ]:
frec_abs = metro["weather_main"].value_counts()
frec_rel = metro["weather_main"].value_counts(normalize=True) * 100

tabla_clima = pd.DataFrame({
    "Frecuencia Absoluta": frec_abs,
    "frecuencia Relativa (%)": frec_rel.round(2)
})

print("frecuencia de tipos de clima: ")
print(tabla_clima)

print("")
print("Registros en dias festivos:")
print(metro["holiday"].value_counts(dropna=False).head(10))

## Covarianza y correlacion

In [ ]:
matriz_covarianza = metro[columnas_numericas].cov()
print("Matriz de covarianza:")
print(matriz_covarianza)

matriz_correlacion = metro[columnas_numericas].corr()
print("")
print("matriz de correlacion")
print(matriz_correlacion)

plt.figure(figsize=(8, 6))
sns.heatmap(matriz_correlacion, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, square=True)
plt.title("Mapa de calor de correlacion de variables numericas")
plt.tight_layout()
plt.show()

## Estadistica por clima

In [ ]:
estadistica_agrupada = metro.groupby("weather_main")["traffic_volume"].agg(
    Conteo="count",
    Media="mean",
    Desviacion_Estandar="std",
    Mediana="median",
    Minimo="min",
    Maximo="max"
).sort_values(by="conteo", ascending=False)

estadistica_agrupada

## Graficas

Histogramas, curvas de densidad y diagramas de caja para revisar las variables numericas.

In [ ]:
metro[columnas_numericas].hist(figsize=(12, 10), bins=20, color="steelblue", edgecolor="white")
plt.suptitle("histogramas de Variables Numericas", fontsize=14)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(metro["temp"], bins=30, kde=True, color="teal", edgecolor="white", ax=axes[0])
axes[0].set_title("Distribucion de Temperatura con KDE")
axes[0].set_xlabel("Temperatura (K)")
axes[0].set_ylabel("frecuencia")

sns.histplot(metro["traffic_volume"], bins=30, kde=True, color="darkorange", edgecolor="white", ax=axes[1])
axes[1].set_title("Distribucion de Volumen de Trafico con KDE")
axes[1].set_xlabel("volumen de Trafico")
axes[1].set_ylabel("frecuencia")

plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
sns.kdeplot(metro["temp"], fill=True, color="purple", alpha=0.4)
plt.xlabel("temperatura (K)")
plt.ylabel("Densidad")
plt.title("Curva de Densidad de Kernel (KDE) - Temperatura")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=metro, y="temp", ax=axes[0], color="lightblue")
axes[0].set_title("Diagrama de Caja - Temperatura")

sns.boxplot(data=metro, y="traffic_volume", ax=axes[1], color="lightgreen")
axes[1].set_title("Diagrama de Caja - Volumen de Trafico")

plt.tight_layout()
plt.show()

## Conclusiones

- traffic_volume tiene un promedio cercano a 3290 y una variacion amplia.
- temp tiene algunos valores bajos que afectan la distribucion.
- Clouds y Clear concentran la mayor parte de los registros.
- rain_1h y snow_1h tienen muchos valores en cero.